# 12주차 복습과제 - Swin Transformer


---
## 목차
1. 상대적 위치 편향 계산 (예제 10.10 ~ 10.12)
2. 모델 실습 - 사전 학습된 Swin Transformer 불러오기 (예제 10.13)
3. 패치 임베딩 모듈 확인 (예제 10.14)
4. 스윈 트랜스포머 블록 구조 확인 (예제 10.15 ~ 10.16)
5. W-MSA, SW-MSA 모듈 실행 (예제 10.17)
6. 패치 병합 (예제 10.18)
7. 스윈 트랜스포머 모델 학습 (예제 10.19)

---
## 1. 상대적 위치 편향(Relative Position Bias) 계산

> 이미지 패치 간의 상대적 거리를 X축, Y축 두 방향으로 표현하고,  
> 이를 어텐션 수식에 편향(bias)으로 추가하는 Swin Transformer의 핵심 메커니즘입니다.
>
> - `window_size = 2`이면 2×2 윈도우 → 패치 4개
> - 상대적 위치 편향 수식: $\text{Attention}(Q,K,V) = \text{SoftMax}(QK^T/\sqrt{d} + B)V$

In [1]:
# 예제 10.10 상대적 위치 편향 계산
# window_size=2인 윈도우 내 패치들의 X, Y 상대적 위치 행렬 ①②를 구한다.

import torch

window_size = 2
coords_h = torch.arange(window_size)
coords_w = torch.arange(window_size)
coords = torch.stack(torch.meshgrid([coords_h, coords_w], indexing="ij"))
coords_flatten = torch.flatten(coords, 1)
relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]

print(relative_coords)  # ①, ②번 연산 과정
print(relative_coords.shape)

tensor([[[ 0,  0, -1, -1],
         [ 0,  0, -1, -1],
         [ 1,  1,  0,  0],
         [ 1,  1,  0,  0]],

        [[ 0, -1,  0, -1],
         [ 1,  0,  1,  0],
         [ 0, -1,  0, -1],
         [ 1,  0,  1,  0]]])
torch.Size([2, 4, 4])


In [2]:
# 예제 10.11 X, Y축에 대한 위치 행렬
# relative_coords를 X, Y 좌표로 분리한 후 offset(③번)을 더하고 x_coords에 스케일(④번)을 적용한다.
# 최종적으로 x_coords + y_coords로 상대적 위치 좌표 행렬(⑤번)을 만든다.

x_coords = relative_coords[0, :, :]
y_coords = relative_coords[1, :, :]

x_coords += window_size - 1  # X축에 대한 ③번 연산 과정
y_coords += window_size - 1  # Y축에 대한 ③번 연산 과정
x_coords *= 2 * window_size - 1  # ④번 연산 과정
print(f"X축에 대한 행렬:\n{x_coords}\n")
print(f"Y축에 대한 행렬:\n{y_coords}\n")

X축에 대한 행렬:
tensor([[3, 3, 0, 0],
        [3, 3, 0, 0],
        [6, 6, 3, 3],
        [6, 6, 3, 3]])

Y축에 대한 행렬:
tensor([[1, 0, 1, 0],
        [2, 1, 2, 1],
        [1, 0, 1, 0],
        [2, 1, 2, 1]])



In [3]:
# 예제 10.12 X, Y축에 대한 상대적 위치 좌표 변환
# 상대적 위치 좌표를 인덱스로 변환해 헤드별 relative_position_bias_table에서 편향 값을 불러온다.
# 결과 텐서 shape: [num_heads, window_size*window_size, window_size*window_size]

relative_position_index = x_coords + y_coords  # ⑤번 연산 과정
print(f"X, Y축에 대한 위치 행렬:\n{relative_position_index}")

num_heads = 1
relative_position_bias_table = torch.Tensor(
    torch.zeros((2 * window_size - 1) * (2 * window_size - 1), num_heads)
)

relative_position_bias = relative_position_bias_table[relative_position_index.view(-1)]
relative_position_bias = relative_position_bias.view(
    window_size * window_size, window_size * window_size, -1
)
print(relative_position_bias.shape)

X, Y축에 대한 위치 행렬:
tensor([[4, 3, 1, 0],
        [5, 4, 2, 1],
        [7, 6, 4, 3],
        [8, 7, 5, 4]])
torch.Size([4, 4, 1])


---
## 2. 사전 학습된 Swin Transformer 모델 불러오기

> **모델**: `microsoft/swin-tiny-patch4-window7-224`  
> - ImageNet-1k (약 1,300만 장, 1,000 클래스)로 사전 학습  
> - 입력 이미지: 224×224, 패치 크기: 4×4, 로컬 윈도우: 7×7  
> - 구조: `swin`(embeddings + encoder) + `classifier`
>
> 아래 코드에서 `train_dataset`은 ViT 예제(예제 10.1~10.3)의 FashionMNIST 데이터셋을 사용합니다.

In [4]:
# 예제 10.13 사전 학습된 스윈 트랜스포머 모델
# HuggingFace transformers의 SwinForImageClassification을 불러와 모델 전체 구조를 출력한다.
# swin: patch partitioning + linear embedding + 4개 스테이지(0~3) + layernorm + pooler
# classifier: 최종 클래스 분류 헤드

from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_dataset = datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)

from transformers import SwinForImageClassification

model = SwinForImageClassification.from_pretrained(
    pretrained_model_name_or_path="microsoft/swin-tiny-patch4-window7-224",
    num_labels=len(train_dataset.classes),
    id2label={idx: label for label, idx in train_dataset.class_to_idx.items()},
    label2id=train_dataset.class_to_idx,
    ignore_mismatched_sizes=True
)

for main_name, main_module in model.named_children():
    print(main_name)
    for sub_name, sub_module in main_module.named_children():
        print("└", sub_name)
        for ssub_name, ssub_module in sub_module.named_children():
            print("|  └", ssub_name)
            for sssub_name, sssub_module in ssub_module.named_children():
                if sssub_name == "projection":
                    print("|  |  └", sssub_name, sssub_module)
                else:
                    print("|  |  └", sssub_name)

100%|██████████| 26.4M/26.4M [00:01<00:00, 19.9MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 340kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.58MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 25.4MB/s]
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/113M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/233 [00:00<?, ?it/s]

SwinForImageClassification LOAD REPORT from: microsoft/swin-tiny-patch4-window7-224
Key               | Status   |                                                                                         
------------------+----------+-----------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([10])          
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([10, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


swin
└ embeddings
|  └ patch_embeddings
|  |  └ projection Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
|  └ norm
|  └ dropout
└ encoder
|  └ layers
|  |  └ 0
|  |  └ 1
|  |  └ 2
|  |  └ 3
└ layernorm
└ pooler
classifier


---
## 3. 패치 임베딩(Patch Embedding) 모듈 확인

> Swin Transformer의 패치 임베딩은 `Conv2d(3, 96, kernel_size=(4,4), stride=(4,4))`로 수행됩니다.  
> 224×224 이미지에 적용하면 → $\frac{224-4}{4}+1=56$ → 56×56 텐서 → 3,136개 패치 생성 (채널 96)  
> ViT는 [CLS] 토큰 포함 197개 패치/768채널인 반면, Swin은 [CLS] 토큰 없이 평균값으로 분류합니다.

In [5]:
# 예제 10.14 패치 임베딩 모듈
# patch_embeddings 모듈만 단독 실행해 입력/출력 차원을 확인한다.
# 입력: [32, 3, 224, 224] → 출력: [32, 3136, 96]

from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from transformers import AutoImageProcessor

image_processor = AutoImageProcessor.from_pretrained("microsoft/swin-tiny-patch4-window7-224")

def transform(examples):
    examples["pixel_values"] = [
        image_processor(img.convert("RGB"), return_tensors="pt")["pixel_values"].squeeze()
        for img in examples["image"]
    ]
    return examples

train_dataset = datasets.FashionMNIST(root="./data", train=True, download=True)

# HuggingFace datasets 형식이 아니라 torchvision이면 아래처럼 간단하게
transform_fn = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
])

train_dataset = datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform_fn)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

batch = next(iter(train_dataloader))
print("이미지 차원 :", batch[0].shape)
patch_emb_output, shape = model.swin.embeddings.patch_embeddings(batch[0])


print("모듈:", model.swin.embeddings.patch_embeddings)
print("패치 임베딩 차원 :", patch_emb_output.shape)

preprocessor_config.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


이미지 차원 : torch.Size([32, 3, 224, 224])
모듈: SwinPatchEmbeddings(
  (projection): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
)
패치 임베딩 차원 : torch.Size([32, 3136, 96])


---
## 4. 스윈 트랜스포머 블록 구조 확인

> **스테이지 구조**: `blocks`(SwinLayer 반복) + `downsample`(SwinPatchMerging)  
> **SwinLayer 내부 순서**: LayerNorm → W-MSA(또는 SW-MSA) → LayerNorm → MLP  
> - `SwinLayer-0` (blocks[0]): **W-MSA** (윈도우 내 어텐션)
> - `SwinLayer-1` (blocks[1]): **SW-MSA** (쉬프트된 윈도우 어텐션)
>
> MLP는 Linear(96→384) + GELU + Linear(384→96)으로 구성됩니다.

In [6]:
# 예제 10.15 스윈 트랜스포머 블록
# 첫 번째 스테이지(encoder.layers[0])의 전체 서브모듈 계층 구조를 출력한다.
# blocks(W-MSA, SW-MSA) + downsample(패치 병합) + norm으로 구성됨을 확인한다.

for main_name, main_module in model.swin.encoder.layers[0].named_children():
    print(main_name)
    for sub_name, sub_module in main_module.named_children():
        print("└", sub_name)
        for ssub_name, ssub_module in sub_module.named_children():
            print("|  └", ssub_name)

blocks
└ 0
|  └ layernorm_before
|  └ attention
|  └ drop_path
|  └ layernorm_after
|  └ intermediate
|  └ output
└ 1
|  └ layernorm_before
|  └ attention
|  └ drop_path
|  └ layernorm_after
|  └ intermediate
|  └ output
downsample
└ reduction
└ norm


In [7]:
# 예제 10.16 SwinLayer 구조
# blocks[0] (W-MSA 담당 SwinLayer)의 상세 구조를 출력한다.
# layernorm_before / attention(SwinSelfAttention: Q,K,V linear) / drop_path
# layernorm_after / intermediate(GELU 활성화) / output

print(model.swin.encoder.layers[0].blocks[0])

SwinLayer(
  (layernorm_before): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
  (attention): SwinAttention(
    (self): SwinSelfAttention(
      (query): Linear(in_features=96, out_features=96, bias=True)
      (key): Linear(in_features=96, out_features=96, bias=True)
      (value): Linear(in_features=96, out_features=96, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (output): SwinSelfOutput(
      (dense): Linear(in_features=96, out_features=96, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
  )
  (drop_path): Identity()
  (layernorm_after): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
  (intermediate): SwinIntermediate(
    (dense): Linear(in_features=96, out_features=384, bias=True)
    (intermediate_act_fn): GELUActivation()
  )
  (output): SwinOutput(
    (dense): Linear(in_features=384, out_features=96, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
)


---
## 5. W-MSA / SW-MSA 모듈 실행

> - **W-MSA** (`blocks[0]`): 윈도우 내부에서만 어텐션 수행 → 계산 효율적
> - **SW-MSA** (`blocks[1]`): 윈도우를 절반 크기만큼 이동(shift)해 경계 간 정보 교환
> - 두 모듈의 입출력 차원은 모두 동일: `[32, 3136, 96]`

In [8]:
# 예제 10.17 W-MSA, SW-MSA 모듈
# patch_emb_output을 W-MSA → SW-MSA 순서로 통과시켜 각 출력 차원을 확인한다.
# 입력/W-MSA 출력/SW-MSA 출력 모두 [32, 3136, 96]으로 동일한 하이퍼파라미터 구성임을 확인.

print("패치 임베딩 차원 :", patch_emb_output.shape)

W_MSA = model.swin.encoder.layers[0].blocks[0]
SW_MSA = model.swin.encoder.layers[0].blocks[1]

W_MSA_output = W_MSA(patch_emb_output, W_MSA.input_resolution)[0]
SW_MSA_output = SW_MSA(W_MSA_output, SW_MSA.input_resolution)[0]

print("W-MSA 결과 차원 :", W_MSA_output.shape)
print("SW-MSA 결과 차원 :", SW_MSA_output.shape)

패치 임베딩 차원 : torch.Size([32, 3136, 96])
W-MSA 결과 차원 : torch.Size([32, 3136, 96])
SW-MSA 결과 차원 : torch.Size([32, 3136, 96])


---
## 6. 패치 병합(Patch Merging)

> `SwinPatchMerging`은 스테이지 간 해상도를 절반으로 줄이는 다운샘플링 역할을 합니다.  
> - 3,136(56×56)개 패치 → 784(28×28)개 패치  
> - 채널: 96×4 = 384 → Linear로 절반(192)으로 축소  
> - 최종 4개 스테이지 통과 후: `[N, 49(7×7), 768]` → Pooling → 클래스 예측

In [9]:
# 예제 10.18 패치 병합
# encoder.layers[0].downsample (SwinPatchMerging) 모듈을 확인하고
# SW-MSA 출력에 적용해 [32, 784, 192]로 변환되는 것을 확인한다.

patch_merge = model.swin.encoder.layers[0].downsample
print("patch_merge 모듈 :", patch_merge)

output = patch_merge(SW_MSA_output, patch_merge.input_resolution)
print("patch_merge 결과 차원 :", output.shape)

patch_merge 모듈 : SwinPatchMerging(
  (reduction): Linear(in_features=384, out_features=192, bias=False)
  (norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
)
patch_merge 결과 차원 : torch.Size([32, 784, 192])


---
## 7. 스윈 트랜스포머 모델 학습 (FashionMNIST Fine-tuning)

> ViT 미세 조정 코드(예제 10.8)에서 모델과 output_dir만 변경합니다.  
> **하이퍼파라미터**: lr=1e-5, batch=16, epochs=3, weight_decay=0.001, seed=7  
> **결과 (표 10.3)**:
>
> | Epoch | Training Loss | Validation Loss | F1     |
> |-------|--------------|-----------------|--------|
> | 1     | 0.3720       | 0.3046          | 0.8985 |
> | 2     | 0.2579       | 0.2688          | 0.9138 |
> | 3     | 0.2607       | 0.2416          | 0.9159 |
>
> 테스트 정확도 약 **91.6%** — ViT 대비 F1은 유사하나 검증 손실이 0.4357 → 0.2416으로 크게 개선,  
> 수렴 속도도 더 빠름. 'Shirt' 오분류 문제가 완화됨.

In [10]:
# 예제 10.19 스윈 트랜스포머 모델 학습
# ViT 예제 10.8과 동일한 Trainer 기반 학습 코드.
# 변경점: model_init에서 SwinForImageClassification 사용, output_dir 변경,
#         image_processor를 swin-tiny 체크포인트로 변경.

from transformers import SwinForImageClassification, TrainingArguments, Trainer, AutoImageProcessor

# 데이터셋, 전처리 코드는 예제 10.1, 10.2, 10.3 참고

def model_init(classes, class_to_idx):
    model = SwinForImageClassification.from_pretrained(
        pretrained_model_name_or_path="microsoft/swin-tiny-patch4-window7-224",
        num_labels=len(classes),
        id2label={idx: label for label, idx in class_to_idx.items()},
        label2id=class_to_idx,
        ignore_mismatched_sizes=True
    )
    return model


image_processor = AutoImageProcessor.from_pretrained(
    pretrained_model_name_or_path="microsoft/swin-tiny-patch4-window7-224"
)


args = TrainingArguments(
    output_dir="../models/Swin-FashionMNIST",
    save_strategy="epoch",
    eval_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.001,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=125,
    remove_unused_columns=False,
    seed=7
)


In [12]:
!pip install evaluate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00


In [13]:
import numpy as np
import evaluate
from torch.utils.data import Dataset as TorchDataset
from transformers import DefaultDataCollator

# 래퍼 Dataset
class FashionDataset(TorchDataset):
    def __init__(self, dataset):
        self.dataset = dataset
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        return {"pixel_values": image, "labels": label}

test_dataset = datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform_fn)

train_dataset_hf = FashionDataset(train_dataset)
test_dataset_hf = FashionDataset(test_dataset)

# 모델 초기화
model = model_init(train_dataset.classes, train_dataset.class_to_idx)

# 평가 지표
metric = evaluate.load("f1")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels, average="weighted")

# Trainer 생성 및 학습
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset_hf,
    eval_dataset=test_dataset_hf,
    compute_metrics=compute_metrics,
    data_collator=DefaultDataCollator(),
)

trainer.train()

Loading weights:   0%|          | 0/233 [00:00<?, ?it/s]

SwinForImageClassification LOAD REPORT from: microsoft/swin-tiny-patch4-window7-224
Key               | Status   |                                                                                         
------------------+----------+-----------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([10])          
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([10, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Epoch,Training Loss,Validation Loss,F1
1,0.188198,0.256331,0.916134
2,0.169355,0.205157,0.936623
3,0.127209,0.191077,0.942034


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=11250, training_loss=0.21260328250461155, metrics={'train_runtime': 2867.3136, 'train_samples_per_second': 62.777, 'train_steps_per_second': 3.924, 'total_flos': 4.47507814957056e+18, 'train_loss': 0.21260328250461155, 'epoch': 3.0})

In [14]:
# 학습 결과 확인 - 테스트셋 평가
results = trainer.evaluate(test_dataset_hf)
print(results)

{'eval_loss': 0.1910766065120697, 'eval_f1': 0.9420335747777182, 'eval_runtime': 81.1878, 'eval_samples_per_second': 123.171, 'eval_steps_per_second': 7.698, 'epoch': 3.0}
